# Lesson 10: logistic regression on the term spread, a warning at p ≥ 0.2

The term spread is the 10-year Treasury yield minus the 3-month bill rate. A month gets a warning when the probability of a US recession 12 months later is at least $t = 0.2$. A missed recession costs four times as much as a false alarm, so a warning has the lower expected cost from $p = 1/(1 + 4) = 0.2$ on. The weights are fitted on the months April 1953 to December 1999, and the warnings are counted on January 2000 to September 2024.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

months = pd.read_csv("https://digital-ai-finance.github.io/teaching-msc-data-science/data/2026-09-30-us-recession-monthly.csv")
train, test = months[months.split == "train"], months[months.split == "test"]
model = LogisticRegression(C=np.inf).fit(train[["spread"]], train.recession_in_12m)  # C=inf: no penalty
print(f"score z = {model.intercept_[0]:.3f} {model.coef_[0, 0]:+.2f} * spread")

grid = pd.DataFrame({"spread": np.linspace(-4, 5, 200)})
plt.scatter(train.spread, train.recession_in_12m, s=6, alpha=0.3, label="training months")
plt.plot(grid.spread, model.predict_proba(grid)[:, 1], color="black", label="probability p")
plt.axhline(0.2, ls="--", color="grey", label="threshold t = 0.2")
plt.xlabel("term spread (percentage points)")
plt.ylabel("probability of a recession 12 months later")
plt.legend(loc="center right")
plt.show()

In [ ]:
t = 0.2  # a miss costs four false alarms: t = 1/(1 + 4)
p = model.predict_proba(test[["spread"]])[:, 1]
tn, fp, fn, tp = confusion_matrix(test.recession_in_12m, p >= t).ravel()
print(f"hits {tp}, misses {fn}, false alarms {fp}, calm months left calm {tn}")
print(f"cost at a miss of 24, a false alarm of 6: {24 * fn + 6 * fp}")
print(f"AUC on the test months: {roc_auc_score(test.recession_in_12m, p):.3f}")

Change `t = 0.2` to `t = 0.5` and run the cell again. At 0.5 the 28 pre-recession test months get no warning, and the cost grows from 618 to 810.